# 📚 Défi quotidien : Analyse textuelle de livres de Lewis Carroll

## Corpus
| # | Livre | URL |
|---|-------|-----|
| 1 | Alice's Adventures in Wonderland | https://www.gutenberg.org/cache/epub/11/pg11.txt |
| 2 | Through the Looking-Glass | https://www.gutenberg.org/cache/epub/12/pg12.txt |
| 3 | A Tangled Tale | https://www.gutenberg.org/cache/epub/29042/pg29042.txt |

## Plan
1. Prétraitement (chargement, nettoyage, tokenisation, stopwords, stemming, lemmatisation, POS, NER)
2. Analyse — Nuage de mots, Bag of Words (BoW)
3. TF-IDF pour identifier les mots vraiment discriminants

---
## 📦 Installation & Imports

In [ ]:
%pip install --quiet nltk spacy wordcloud matplotlib scikit-learn requests

In [ ]:
import re
import requests
import nltk
import spacy
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Download required NLTK resources
for resource in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger",
                 "averaged_perceptron_tagger_eng", "maxent_ne_chunker",
                 "maxent_ne_chunker_tab", "words"]:
    nltk.download(resource, quiet=True)

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download as spacy_download
    spacy_download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("✅ All libraries ready")
print("spaCy pipeline:", nlp.pipe_names)

URLS = [
    "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "https://www.gutenberg.org/cache/epub/12/pg12.txt",
    "https://www.gutenberg.org/cache/epub/29042/pg29042.txt",
]
TITLES = [
    "Alice's Adventures in Wonderland",
    "Through the Looking-Glass",
    "A Tangled Tale",
]

---
# PARTIE 1 — Prétraitement du texte

## Étape 1 — `load_texts()` : chargement et nettoyage

La fonction :
- Télécharge chaque texte via `requests`
- Supprime les non-mots avec une regex `[^a-zA-Z\s]`
- Extrait uniquement la partie entre `*** START` et `*** END` (ignorer les mentions légales Project Gutenberg)

In [ ]:
def load_texts(urls: list) -> list:
    """
    Download books from Project Gutenberg, strip Gutenberg header/footer,
    and clean non-word characters using regex.

    Args:
        urls: list of .txt URLs

    Returns:
        corpus: list of cleaned text strings (one per book)
    """
    corpus = []
    for url in urls:
        print(f"  Downloading: {url}")
        response = requests.get(url)
        response.encoding = "utf-8"
        raw = response.text

        # ── Strip Gutenberg header and footer ────────────────────────────
        # Keep only the literary content between START and END markers
        start_match = re.search(r'\*{3}\s*START[^\n]*\n', raw, re.IGNORECASE)
        end_match   = re.search(r'\*{3}\s*END[^\n]*\n',   raw, re.IGNORECASE)

        if start_match and end_match:
            raw = raw[start_match.end(): end_match.start()]

        # ── Remove non-word characters (keep letters and spaces) ─────────
        cleaned = re.sub(r"[^a-zA-Z\s]", " ", raw)
        # Collapse multiple spaces
        cleaned = re.sub(r"\s+", " ", cleaned).strip()

        corpus.append(cleaned)

    return corpus


print("Loading corpus...")
corpus = load_texts(URLS)
print(f"\n✅ {len(corpus)} books loaded")

## Étape 2 — Aperçu : 200 premiers caractères

In [ ]:
for title, text in zip(TITLES, corpus):
    print(f"\n{'='*60}")
    print(f"  📖 {title}")
    print(f"{'='*60}")
    print(text[:200], "...")
    print(f"  Total characters: {len(text):,}")

## Étape 3 — Tokenisation : 150 premiers tokens

In [ ]:
# Tokenize each book with NLTK word_tokenize
tokenized_corpus = [word_tokenize(text.lower()) for text in corpus]

for title, tokens in zip(TITLES, tokenized_corpus):
    print(f"\n📖 {title}")
    print(f"   Total tokens : {len(tokens):,}")
    print(f"   First 150    : {tokens[:150]}")

## Étape 4 — Suppression des mots vides (stopwords)

In [ ]:
stop_words = set(stopwords.words("english"))
print(f"Stopwords loaded: {len(stop_words)} words")
print(f"Sample: {sorted(list(stop_words))[:15]}")

# Remove stopwords from each tokenized book
filtered_corpus = [
    [token for token in tokens if token not in stop_words and token.isalpha()]
    for tokens in tokenized_corpus
]

# Verify removal by searching for common stopwords
check_words = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves"]
print("\n--- Stopword removal verification ---")
for title, tokens in zip(TITLES, filtered_corpus):
    print(f"\n📖 {title}")
    print(f"   Tokens after filtering: {len(tokens):,}")
    for sw in check_words:
        count = tokens.count(sw)
        status = "✅ removed" if count == 0 else f"⚠️  still appears {count}x"
        print(f"   '{sw}': {status}")

## Étape 5 — Stemming avec `PorterStemmer`

Le stemming coupe les suffixes des mots pour réduire les variantes morphologiques à une racine commune (qui n'est pas forcément un vrai mot).

In [ ]:
stemmer = PorterStemmer()

# Apply stemming to each filtered book
stemmed_corpus = [
    [stemmer.stem(token) for token in tokens]
    for tokens in filtered_corpus
]

print("=== First 50 stemmed tokens ===")
for title, stemmed in zip(TITLES, stemmed_corpus):
    print(f"\n📖 {title}")
    print(stemmed[:50])

## Étape 6 — Lemmatisation avec spaCy `en_core_web_sm`

La lemmatisation retourne le **lemme** (forme canonique du dictionnaire) grâce au modèle linguistique spaCy. Ex : `running → run`, `mice → mouse`.

In [ ]:
def lemmatize_tokens(tokens: list, batch_size: int = 5000) -> list:
    """
    Lemmatize a list of tokens using spaCy.
    Processes in batches to handle long texts efficiently.
    The lemma is accessed via token.lemma_ attribute.
    """
    lemmas = []
    # spaCy works on raw text; we join tokens and re-process
    text = " ".join(tokens)
    # Increase max_length for long books
    nlp.max_length = max(nlp.max_length, len(text) + 10)
    doc = nlp(text)
    lemmas = [token.lemma_ for token in doc if token.is_alpha]
    return lemmas


print("Running spaCy lemmatization (this may take ~1 min per book)...")
lemmatized_corpus = []
for title, tokens in zip(TITLES, filtered_corpus):
    print(f"  Processing: {title}...")
    lemmas = lemmatize_tokens(tokens)
    lemmatized_corpus.append(lemmas)
    print(f"  ✅ Done — {len(lemmas):,} lemmas")

print("\n=== First 50 lemmatized tokens ===")
for title, lemmas in zip(TITLES, lemmatized_corpus):
    print(f"\n📖 {title}")
    print(lemmas[:50])

## Étape 7 — Analyse : Stemming vs Lemmatisation

In [ ]:
# Compare stemming vs lemmatization on a selection of words
compare_words = ["running", "better", "mice", "wonderfully", "said",
                 "cried", "having", "looked", "playing", "goes"]

print(f"{'Word':<15} {'Stem (Porter)':<20} {'Lemma (spaCy)':<20}")
print("-" * 55)
for word in compare_words:
    stem  = stemmer.stem(word)
    lemma = nlp(word)[0].lemma_
    print(f"{word:<15} {stem:<20} {lemma:<20}")

print("""
=== Analysis ===

Stemming (PorterStemmer)
  - RULE-BASED: applies fixed suffix-stripping rules (e.g., -ing, -ed, -ly)
  - FAST: no linguistic model needed
  - LOSSY: often produces non-words (e.g., "wonderfully" → "wonder")
  - AGGRESSIVE: "better" stays "better", "mice" stays "mice"

Lemmatisation (spaCy)
  - DICTIONARY-BASED: returns the canonical dictionary form (lemma)
  - CONTEXT-AWARE: uses POS tagging internally
  - CONSERVATIVE: always returns a valid word (e.g., "mice" → "mouse", "better" → "well")
  - SLOWER: requires a pre-trained language model

=> For NLP tasks (BoW, TF-IDF), lemmatisation is preferred because it
   preserves meaning while reducing morphological variation accurately.
""")

## Étape 8 — Étiquetage POS avec NLTK

In [ ]:
print("=== POS Tagging (first 30 tokens per book) ===")
pos_corpus = []

for title, tokens in zip(TITLES, filtered_corpus):
    tags = pos_tag(tokens)
    pos_corpus.append(tags)

    print(f"\n📖 {title}")
    print(tags[:30])

    # Count most common POS tags
    tag_counts = Counter(tag for _, tag in tags)
    print(f"\n   Top 8 POS tags:")
    for tag, count in tag_counts.most_common(8):
        print(f"   {tag:<8} → {count:>5}x")

# Tag legend
print("""
--- Common Penn Treebank POS Tags ---
NN=Noun  NNP=Proper Noun  VBD=Verb(past)  VBZ=Verb(3sg)
JJ=Adjective  RB=Adverb  IN=Preposition  DT=Determiner
""")

## Étape 9 — Reconnaissance d'entités nommées (NER) avec NLTK

In [ ]:
from nltk import ne_chunk, Tree

def extract_entities(pos_tags: list) -> list:
    """Extract named entities from POS-tagged tokens using NLTK ne_chunk."""
    tree     = ne_chunk(pos_tags)
    entities = []
    for subtree in tree:
        if isinstance(subtree, Tree):
            entity_text  = " ".join(word for word, tag in subtree.leaves())
            entity_label = subtree.label()
            entities.append((entity_text, entity_label))
    return entities


print("=== Named Entity Recognition (NER) ===")
for title, pos_tags in zip(TITLES, pos_corpus):
    # Use a limited window for speed (first 2000 tokens)
    entities = extract_entities(pos_tags[:2000])
    print(f"\n📖 {title}")
    print(f"   Entities found (in first 2000 tokens): {len(entities)}")

    # Group by entity type
    by_type = {}
    for text, label in entities:
        by_type.setdefault(label, []).append(text)

    for label, items in sorted(by_type.items()):
        unique_items = list(dict.fromkeys(items))[:8]  # deduplicate, max 8
        print(f"   {label:<12}: {unique_items}")

---
# PARTIE 2 — Analyse du texte

## Étape 1 — Nuages de mots (Word Clouds)

In [ ]:
COLORS = ["Blues", "Greens", "Oranges"]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, title, lemmas, cmap in zip(axes, TITLES, lemmatized_corpus, COLORS):
    # Join lemmas into a single string for WordCloud
    text_for_cloud = " ".join(lemmas)

    wc = WordCloud(
        width=800,
        height=500,
        background_color="white",
        colormap=cmap,
        max_words=120,
        collocations=False,    # avoid duplicate bigrams
        stopwords=stop_words
    ).generate(text_for_cloud)

    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)

plt.suptitle("Word Clouds — Lewis Carroll Books (lemmatized)",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("wordclouds.png", dpi=150, bbox_inches="tight")
plt.show()
print("Word clouds saved → wordclouds.png")

## Étape 2 — Bag of Words (BoW) : 5 mots les plus fréquents

Nous utilisons les textes **lemmatisés** (meilleur équilibre entre normalisation et lisibilité).

In [ ]:
# Join lemmas back into strings for CountVectorizer
lemma_texts = [" ".join(lemmas) for lemmas in lemmatized_corpus]

vectorizer = CountVectorizer(stop_words="english")
bow_matrix = vectorizer.fit_transform(lemma_texts)

feature_names = vectorizer.get_feature_names_out()
print(f"Vocabulary size : {len(feature_names):,} unique words")
print(f"BoW matrix shape: {bow_matrix.shape}  (documents × words)")

print("\n=== Top 5 words per book (BoW) ===")
top5_bow = {}
for i, title in enumerate(TITLES):
    row    = bow_matrix[i].toarray().flatten()
    top5_i = sorted(zip(feature_names, row), key=lambda x: x[1], reverse=True)[:5]
    top5_bow[title] = top5_i
    print(f"\n📖 {title}")
    for word, freq in top5_i:
        print(f"   '{word}' → {int(freq)}x")

## Étape 3 — Structure du BoW : document, index, fréquence

In [ ]:
# Display the sparse BoW matrix in (doc_id, word_index) → frequency format
print("BoW sparse representation (document_id, word_index) → count")
print("Format: (doc #, word_index)  count")
print("-" * 50)

cx = bow_matrix.tocoo()
for doc_id, word_idx, count in zip(cx.row, cx.col, cx.data):
    if count > 50:  # only show high-frequency entries for readability
        print(f"  (doc={doc_id} [{TITLES[doc_id][:25]}...], idx={word_idx:>5}, word='{feature_names[word_idx]:>12}') → {count}x")

print("""
Reading the BoW matrix:
  • doc #   : which book (0=Alice, 1=Looking-Glass, 2=Tangled Tale)
  • word idx: position of the word in the vocabulary (column index)
  • count   : how many times that word appears in that document
""")

## Étape 4 — Diagrammes circulaires : Top 5 mots par livre (BoW)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

palette = plt.cm.Set2.colors

for ax, title in zip(axes, TITLES):
    words  = [w for w, _ in top5_bow[title]]
    counts = [int(c) for _, c in top5_bow[title]]
    colors = palette[:len(words)]

    wedges, _ = ax.pie(
        counts,
        colors=colors,
        startangle=140,
        wedgeprops=dict(edgecolor="white", linewidth=1.5)
    )
    ax.set_title(title, fontsize=11, fontweight="bold", pad=12)

    # Legend with word + count
    legend_labels = [f"'{w}' ({c:,}x)" for w, c in zip(words, counts)]
    ax.legend(wedges, legend_labels, loc="lower center",
              bbox_to_anchor=(0.5, -0.18), fontsize=9, frameon=False)

plt.suptitle("Top 5 Most Frequent Words per Book — BoW (lemmatized)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("pie_bow.png", dpi=150, bbox_inches="tight")
plt.show()
print("Pie charts saved → pie_bow.png")

## Étape 5 — Analyse des résultats BoW

### Sont-ils informatifs ?

| Mot dominant | Présent dans | Informatif ? |
|-------------|-------------|-------------|
| `alice` | Books 1 & 2 | ✅ Oui — personnage principal |
| `say` / `said` | Tous | ⚠️ Peu — verbe de dialogue générique |
| `little` | Books 1 & 2 | ⚠️ Peu — adjectif très courant |
| `queen` | Book 2 | ✅ Oui — personnage du miroir |
| `knot` / `mad` | Book 3 | ✅ Oui — thème du livre |

### Problème du BoW pur
Les mots les plus fréquents sont souvent des **mots très généraux** (`say`, `little`, `time`, `come`) qui apparaissent dans tous les livres et ne distinguent pas les œuvres entre elles.  
**→ Solution : TF-IDF** (Partie 3)

---
# PARTIE 3 — TF-IDF : résoudre le problème de fréquence

**TF-IDF = Term Frequency × Inverse Document Frequency**

- **TF** : fréquence du mot dans *ce* document (favorise les mots répétés localement)
- **IDF** : pénalise les mots qui apparaissent dans *tous* les documents (ex : `said`, `little`)
- **Résultat** : score élevé = mot spécifique à ce document, score bas = mot banal dans le corpus

## Étape 1 — Créer le TF-IDF

In [ ]:
# min_df=1, max_df=2 : critical for small corpora (3 books)
# min_df=1 : keep words appearing in at least 1 document
# max_df=2 : discard words appearing in more than 2 documents (i.e., all 3)
#            → this naturally penalizes cross-document common words
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=1,
    max_df=2
)
tfidf_matrix = tfidf_vectorizer.fit_transform(lemma_texts)

tfidf_features = tfidf_vectorizer.get_feature_names_out()
print(f"TF-IDF vocabulary size : {len(tfidf_features):,}")
print(f"TF-IDF matrix shape    : {tfidf_matrix.shape}  (documents × words)")

print("\n=== Top 5 words per book (TF-IDF) ===")
top5_tfidf = {}
for i, title in enumerate(TITLES):
    row    = tfidf_matrix[i].toarray().flatten()
    top5_i = sorted(zip(tfidf_features, row), key=lambda x: x[1], reverse=True)[:5]
    top5_tfidf[title] = top5_i
    print(f"\n📖 {title}")
    for word, score in top5_i:
        print(f"   '{word}' → TF-IDF score = {score:.4f}")

## Étape 2 — Diagrammes circulaires TF-IDF

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

palette2 = plt.cm.Set3.colors

for ax, title in zip(axes, TITLES):
    words  = [w for w, _ in top5_tfidf[title]]
    scores = [float(s) for _, s in top5_tfidf[title]]
    colors = palette2[:len(words)]

    wedges, _ = ax.pie(
        scores,
        colors=colors,
        startangle=140,
        wedgeprops=dict(edgecolor="white", linewidth=1.5)
    )
    ax.set_title(title, fontsize=11, fontweight="bold", pad=12)

    legend_labels = [f"'{w}' ({s:.4f})" for w, s in zip(words, scores)]
    ax.legend(wedges, legend_labels, loc="lower center",
              bbox_to_anchor=(0.5, -0.18), fontsize=9, frameon=False)

plt.suptitle("Top 5 Most Relevant Words per Book — TF-IDF (lemmatized)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("pie_tfidf.png", dpi=150, bbox_inches="tight")
plt.show()
print("TF-IDF pie charts saved → pie_tfidf.png")

## Comparaison finale : BoW vs TF-IDF

In [ ]:
print("=" * 70)
print("  COMPARISON: BoW  vs  TF-IDF")
print("=" * 70)

for title in TITLES:
    bow_words   = [w for w, _ in top5_bow[title]]
    tfidf_words = [w for w, _ in top5_tfidf[title]]
    print(f"\n📖 {title}")
    print(f"   BoW   top 5 : {bow_words}")
    print(f"   TF-IDF top 5: {tfidf_words}")

print("""
=== Conclusion ===

BoW favours the most frequent words regardless of their distribution across
the corpus. Words like 'alice', 'say', 'little' dominate because they are
inherently common in narrative text — they tell us little about what makes
each book unique.

TF-IDF penalises words that appear in many documents and rewards words
specific to a single book:
  • Alice in Wonderland  → character names, Wonderland-specific terms
  • Through the Looking-Glass → mirror / chess / Red Queen vocabulary
  • A Tangled Tale       → mathematical / puzzle terminology

=> TF-IDF gives far more discriminative and interpretable results than raw BoW.
   For information retrieval, document classification, and topic analysis,
   TF-IDF is almost always preferred over plain frequency counts.
""")